In [ ]:
"""
=============================================================
 NOTEBOOK 2 — Exploratory Data Analysis
=============================================================
This notebook explores the merged dataset prior to modelling,
examining the relationship between carcass-level features and 
eating quality (EQ) scores across primal cuts.

Analysis includes:
  - Distribution of EQ scores by cut and EQ quartile
  - Continuous feature analysis: scatter plots, box plots, 
    and stacked histograms per feature per EQ quartile
  - Feature importance orientation prior to modelling
  - Excel report generation with per-feature insights

Outputs:
  - Combined feature scatter/histogram grid per cut
  - Individual box plot + summary table per feature
  - Formatted Excel workbook with feature analysis tables
 =============================================================
"""

In [ ]:
# Evaluation of APB Tem & PH time series data on target
# Trial of  CatBoostRegp learning model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import csv
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment
import re
import textwrap
from statsmodels.stats.contingency_tables import mcnemar
import sqlite3
from contextlib import contextmanager
import json
import csv
from fuzzywuzzy import process
from itertools import product
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import mlflow
from catboost import CatBoostRegressor, Pool

In [ ]:
"""
continuous_feature_analysis.py
================================
Generates continuous feature analysis charts and Excel report
for any beef eating quality cut dataset.

Usage
-----
    python continuous_feature_analysis.py

Edit the CONFIG section at the bottom of this file to set:
  - INPUT_PATH   : path to the cut-level CSV file
  - OUTPUT_DIR   : folder where outputs are saved
  - CUT_CODE     : short code used in output filenames (e.g. "FT", "LD")
  - CUT_LABEL    : display name used in chart titles (e.g. "Fillet (FT)")
  - TARGET       : target column name (default "EQ")
  - FEATURES     : dict of {column_name: importance_pct}
  - SHORT_NAMES  : dict of {column_name: display_label}
  - INSIGHTS     : dict of {column_name: [observation_1, observation_2]}

Outputs
-------
  {CUT_CODE}_continuous_features.png        — combined scatter + stacked histogram grid
  {CUT_CODE}_{feature_name}.png             — individual box plot + table per feature
  {CUT_CODE}_Feature_Analysis_Tables.xlsx   — formatted Excel workbook with tables + insights
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import warnings
warnings.filterwarnings('ignore')

# ── Palette ───────────────────────────────────────────────────────────────
BRAND   = "#1B3A5C"
ACCENT  = "#E8622A"
MID     = "#94B8D0"
Q_COLORS      = ["#FF9999", "#FFD966", "#9EC5FE", "#90EE90"]
Q_LABELS_BINS = ["Q1\n(Low EQ)", "Q2", "Q3", "Q4\n(High EQ)"]
Q_LABELS_LEG  = ["Q1 (Low EQ)", "Q2", "Q3", "Q4 (High EQ)"]
Q_LABELS_FULL = ["Q1 (Low EQ)", "Q2", "Q3", "Q4 (High EQ)", "Overall"]
Q_FILLS_HEX   = ["#FFCCCC", "#FFF3CC", "#CCE5FF", "#CCFFDD", "#DDDDDD"]

NAVY   = "1B3A5C"
ORANGE = "E8622A"
LGREY  = "F2F2F2"
WHITE  = "FFFFFF"


# ═══════════════════════════════════════════════════════════════════════════
# CHART HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def make_individual_chart(df, feat, label, imp, target, global_mean, out_path):
    """Individual box plot per EQ quartile with embedded summary table."""
    sub  = df[[feat, target]].dropna()
    x, y = sub[feat].values, sub[target].values
    r, p  = stats.pearsonr(x, y)
    r2    = r ** 2

    q_bins = pd.qcut(y, q=4, labels=Q_LABELS_BINS)
    groups = [x[q_bins == q] for q in Q_LABELS_BINS]
    means  = [np.mean(g) for g in groups]

    fig = plt.figure(figsize=(10, 9), facecolor="white")
    fig.suptitle(
        f"{label}\n"
        f"Importance: {imp:.2f}%   |   r = {r:+.3f}   R² = {r2:.4f}   "
        f"p = {'<0.001' if p < 0.001 else f'{p:.3f}'}   n = {len(sub)}",
        fontsize=11, fontweight="bold", color=BRAND, y=0.99
    )

    ax = fig.add_axes([0.10, 0.38, 0.82, 0.56])
    bp = ax.boxplot(
        groups, patch_artist=True, widths=0.52,
        medianprops=dict(color=ACCENT, lw=2.5),
        whiskerprops=dict(color=BRAND, lw=1.2),
        capprops=dict(color=BRAND, lw=1.2),
        flierprops=dict(marker='o', markersize=3, color="#AAAAAA", alpha=0.5),
    )
    for patch, col in zip(bp["boxes"], Q_COLORS):
        patch.set_facecolor(col)
        patch.set_alpha(0.75)

    ax.plot(range(1, 5), means, 'D', color=BRAND, zorder=6, ms=6, label="Group mean")
    ax.axhline(global_mean, color=ACCENT, lw=1.5, ls="--", alpha=0.8,
               label=f"Global mean {target} = {global_mean:.2f}")

    y_min, y_max = ax.get_ylim()
    for j, (g, gm) in enumerate(zip(groups, means)):
        ax.text(j + 1, y_min + (y_max - y_min) * 0.02,
                f"n={len(g)}\nμ={gm:.2f}", ha='center', va='bottom',
                fontsize=7.5, color=BRAND)

    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(Q_LABELS_BINS, fontsize=9)
    ax.set_xlabel("EQ Quartile", fontsize=10, color=BRAND)
    ax.set_ylabel(label, fontsize=10, color=BRAND)
    ax.tick_params(colors=BRAND, labelsize=8)
    for sp in ax.spines.values():
        sp.set_color("#CCCCCC")
    ax.legend(fontsize=8, framealpha=0.85, loc="upper right")

    # Summary table
    t_rows = [
        [q.replace("\n", " "), len(g), f"{np.mean(g):.3f}", f"{np.median(g):.3f}",
         f"{np.std(g):.3f}", f"{np.min(g):.3f}", f"{np.max(g):.3f}"]
        for q, g in zip(Q_LABELS_BINS, groups)
    ]
    t_rows.append(["Overall", len(x), f"{np.mean(x):.3f}", f"{np.median(x):.3f}",
                   f"{np.std(x):.3f}", f"{np.min(x):.3f}", f"{np.max(x):.3f}"])

    ax_t = fig.add_axes([0.05, 0.01, 0.90, 0.30])
    ax_t.axis('off')
    tbl = ax_t.table(
        cellText=t_rows,
        colLabels=["EQ Quartile", "n", "Mean", "Median", "SD", "Min", "Max"],
        loc='center', cellLoc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.6)
    for j in range(7):
        tbl[0, j].set_facecolor(f"#{NAVY}")
        tbl[0, j].set_text_props(color='white', fontweight='bold')
    for ri, fhex in enumerate(Q_FILLS_HEX):
        for j in range(7):
            tbl[ri + 1, j].set_facecolor(fhex)
            tbl[ri + 1, j].set_alpha(0.6)

    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved {os.path.basename(out_path)}")
    return r, p, r2, len(sub)


def make_combined_chart(df, features, short_names, importance, target,
                        global_mean, cut_label, n_label, out_path):
    """Combined scatter + stacked histogram grid for all continuous features."""
    n     = len(features)
    fig, axes = plt.subplots(n, 2, figsize=(14, n * 4.2), facecolor="white")
    fig.suptitle(
        f"{cut_label} — Continuous Features vs {target}\n"
        f"Individual panellist-level data  |  n={n_label}",
        fontsize=13, fontweight="bold", color=BRAND, y=1.005
    )
    plt.subplots_adjust(hspace=0.52, wspace=0.30)

    # Handle case where n=1 (axes not 2D)
    if n == 1:
        axes = [axes]

    for i, feat in enumerate(features):
        sub2   = df[[feat, target]].dropna()
        x2, y2 = sub2[feat].values, sub2[target].values
        r2v, p2 = stats.pearsonr(x2, y2)
        slope, intercept, *_ = stats.linregress(x2, y2)
        xr    = np.linspace(x2.min(), x2.max(), 200)
        lbl   = short_names[feat]
        imp2  = importance[feat]
        qb    = pd.qcut(y2, q=4, labels=Q_LABELS_BINS)

        ax1 = axes[i][0]
        ax1.scatter(x2, y2, alpha=0.22, s=7, color=MID, linewidths=0)
        ax1.plot(xr, slope * xr + intercept, color=ACCENT, lw=2,
                 zorder=5, label="OLS fit")
        ax1.axhline(global_mean, color=BRAND, lw=1.3, ls="--", alpha=0.75,
                    label=f"Global mean = {global_mean:.2f}")
        ax1.set_xlabel(lbl, fontsize=9, color=BRAND)
        ax1.set_ylabel(target, fontsize=9, color=BRAND)
        ax1.set_title(
            f"{lbl}  ({imp2:.2f}%)\n"
            f"r = {r2v:+.3f}  R² = {r2v**2:.4f}  "
            f"p = {'<0.001' if p2 < 0.001 else f'{p2:.3f}'}  n = {len(sub2)}",
            fontsize=9, color=BRAND, pad=5
        )
        ax1.tick_params(colors=BRAND, labelsize=8)
        for sp in ax1.spines.values():
            sp.set_color("#CCCCCC")
        ax1.legend(fontsize=7.5, framealpha=0.85)

        ax2 = axes[i][1]
        bins   = np.linspace(x2.min(), x2.max(), 30)
        bottom = np.zeros(len(bins) - 1)
        for q_lbl, q_col, q_leg in zip(Q_LABELS_BINS, Q_COLORS, Q_LABELS_LEG):
            counts, _ = np.histogram(x2[qb == q_lbl], bins=bins)
            ax2.bar(bins[:-1], counts, width=np.diff(bins), bottom=bottom,
                    color=q_col, alpha=0.85, label=q_leg, align='edge')
            bottom += counts
        ax2.axvline(np.mean(x2), color=BRAND, lw=1.5, ls="--",
                    label=f"Feature mean = {np.mean(x2):.2f}")
        ax2.set_xlabel(lbl, fontsize=9, color=BRAND)
        ax2.set_ylabel("Count", fontsize=9, color=BRAND)
        ax2.set_title("Count by EQ Quartile (stacked)", fontsize=9, color=BRAND, pad=5)
        ax2.tick_params(colors=BRAND, labelsize=8)
        for sp in ax2.spines.values():
            sp.set_color("#CCCCCC")
        ax2.legend(fontsize=7.5, framealpha=0.85)

    fig.tight_layout(rect=[0, 0, 1, 0.998])
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved {os.path.basename(out_path)}")


# ═══════════════════════════════════════════════════════════════════════════
# EXCEL HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def _fill(h):
    return PatternFill("solid", fgColor=h.lstrip("#"))

_thin = Side(style='thin', color="CCCCCC")
_bdr  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
_ctr  = Alignment(horizontal="center", vertical="center", wrap_text=True)
_lft  = Alignment(horizontal="left",   vertical="center", wrap_text=True)


def _header_row(ws, row, headers, fill_hex):
    for ci, h in enumerate(headers, 1):
        c = ws.cell(row=row, column=ci, value=h)
        c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        c.fill      = _fill(fill_hex)
        c.alignment = _ctr
        c.border    = _bdr


def _data_row(ws, row, values, fill_hex, bold=False):
    for ci, v in enumerate(values, 1):
        c = ws.cell(row=row, column=ci, value=v)
        c.font      = Font(name="Arial", bold=bold, size=9)
        c.fill      = _fill(fill_hex)
        c.alignment = _ctr if ci > 1 else _lft
        c.border    = _bdr


def make_excel(df, features, short_names, importance, insights,
               target, global_mean, cut_label, out_path):
    """Build formatted Excel workbook with per-feature tables and insights."""
    wb = Workbook()
    wb.remove(wb.active)

    # ── Summary sheet ─────────────────────────────────────────────────────
    ws_s = wb.create_sheet(title="Summary", index=0)
    ws_s.column_dimensions['A'].width = 30
    for col in ['B', 'C', 'D', 'E', 'F']:
        ws_s.column_dimensions[col].width = 14

    ws_s.merge_cells("A1:F1")
    c = ws_s["A1"]
    c.value     = f"{cut_label} — Feature Summary  |  Target: {target}"
    c.font      = Font(name="Arial", bold=True, size=12, color="FFFFFF")
    c.fill      = _fill(NAVY)
    c.alignment = _ctr

    _header_row(ws_s, 2, ["Feature", "Importance %", "n", "Pearson r", "R²", "p-value"], NAVY)

    for i, feat in enumerate(features):
        sub2      = df[[feat, target]].dropna()
        rv, pv    = stats.pearsonr(sub2[feat].values, sub2[target].values)
        pv_str    = "<0.001" if pv < 0.001 else round(pv, 3)
        bg        = LGREY if i % 2 == 0 else WHITE
        row_vals  = [short_names[feat], importance[feat], len(sub2),
                     round(rv, 4), round(rv ** 2, 4), pv_str]
        _data_row(ws_s, i + 3, row_vals, bg)

    # ── Per-feature sheets ────────────────────────────────────────────────
    for feat in features:
        label = short_names[feat]
        imp   = importance[feat]
        sub   = df[[feat, target]].dropna()
        x_v, y_v = sub[feat].values, sub[target].values
        r_v, p_v = stats.pearsonr(x_v, y_v)

        sname = (label[:28]
                 .replace("²", "2").replace("/", "_")
                 .replace("(", "").replace(")", "")
                 .replace("%", "pct").replace("-", "_"))
        ws = wb.create_sheet(title=sname)
        ws.column_dimensions['A'].width = 22
        for col in ['B', 'C', 'D', 'E', 'F', 'G']:
            ws.column_dimensions[col].width = 13

        row = 1

        # Title
        ws.merge_cells(f"A{row}:G{row}")
        c           = ws[f"A{row}"]
        c.value     = f"{cut_label}  —  {label}"
        c.font      = Font(name="Arial", bold=True, size=12, color="FFFFFF")
        c.fill      = _fill(NAVY)
        c.alignment = _ctr
        row += 1

        # Stats bar
        stat_str = (
            f"Importance: {imp:.2f}%   |   r = {r_v:+.3f}   |   "
            f"R² = {r_v**2:.4f}   |   "
            f"p = {'<0.001' if p_v < 0.001 else round(p_v, 3)}   |   "
            f"n = {len(sub)}"
        )
        ws.merge_cells(f"A{row}:G{row}")
        c           = ws[f"A{row}"]
        c.value     = stat_str
        c.font      = Font(name="Arial", size=9, color=NAVY)
        c.fill      = _fill("D5E8F0")
        c.alignment = _ctr
        row += 2

        # Data table
        q_bins = pd.qcut(y_v, q=4, labels=["Q1 (Low EQ)", "Q2", "Q3", "Q4 (High EQ)"])
        _header_row(ws, row, ["EQ Quartile", "n", "Mean", "Median", "SD", "Min", "Max"], NAVY)
        row += 1

        for q_lab, fhex in zip(Q_LABELS_FULL[:4], Q_FILLS_HEX[:4]):
            vals = x_v[q_bins == q_lab]
            _data_row(ws, row,
                      [q_lab, len(vals), round(np.mean(vals), 3),
                       round(np.median(vals), 3), round(np.std(vals), 3),
                       round(np.min(vals), 3), round(np.max(vals), 3)],
                      fhex)
            row += 1

        # Overall row
        _data_row(ws, row,
                  ["Overall", len(x_v), round(np.mean(x_v), 3),
                   round(np.median(x_v), 3), round(np.std(x_v), 3),
                   round(np.min(x_v), 3), round(np.max(x_v), 3)],
                  Q_FILLS_HEX[4], bold=True)
        row += 2

        # Global mean note
        ws.merge_cells(f"A{row}:G{row}")
        c           = ws[f"A{row}"]
        c.value     = f"Global mean {target}: {global_mean:.3f}"
        c.font      = Font(name="Arial", size=9, italic=True, color="555555")
        c.alignment = _lft
        row += 2

        # Insights
        ws.merge_cells(f"A{row}:G{row}")
        c           = ws[f"A{row}"]
        c.value     = "Key Observations"
        c.font      = Font(name="Arial", bold=True, size=10, color="FFFFFF")
        c.fill      = _fill(ORANGE)
        c.alignment = _lft
        row += 1

        feat_insights = insights.get(feat, [
            "No observations provided for this feature.",
        ])
        for oi, obs in enumerate(feat_insights, 1):
            ws.merge_cells(f"A{row}:G{row}")
            c           = ws[f"A{row}"]
            c.value     = f"{oi}.  {obs}"
            c.font      = Font(name="Arial", size=9, color="222222")
            c.fill      = _fill("FFF8F0" if oi == 1 else "FFFFFF")
            c.alignment = Alignment(horizontal="left", vertical="center",
                                    wrap_text=True)
            ws.row_dimensions[row].height = 50
            row += 1

        ws.freeze_panes = "A4"

    wb.save(out_path)
    print(f"  Saved {os.path.basename(out_path)}")


# ═══════════════════════════════════════════════════════════════════════════
# MAIN RUNNER
# ═══════════════════════════════════════════════════════════════════════════

def run_analysis(input_path, output_dir, cut_code, cut_label,
                 features, short_names, insights, target="EQ"):
    """
    Run the full continuous feature analysis for one cut.

    Parameters
    ----------
    input_path  : str   Path to the cut-level CSV file
    output_dir  : str   Folder where all outputs are written
    cut_code    : str   Short code for filenames, e.g. "FT"
    cut_label   : str   Display name for chart titles, e.g. "Fillet (FT)"
    features    : dict  {column_name: importance_pct}
    short_names : dict  {column_name: display_label}
    insights    : dict  {column_name: [observation_1, observation_2]}
    target      : str   Target column name (default "EQ")
    """
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  {cut_label}  —  Loading {os.path.basename(input_path)}")
    print(f"{'='*60}")

    df = pd.read_csv(input_path)
    print(f"  Rows: {len(df)}  |  Animals: {df['Eartag'].nunique()}  "
          f"|  Panellists: {df['Username'].nunique()}")

    global_mean = df[target].mean()
    n_label     = f"{len(df):,}"
    feat_list   = list(features.keys())

    # 1. Individual charts
    print(f"\n  Building individual charts...")
    for feat in feat_list:
        safe = (short_names[feat][:50]
                .replace(" ", "_").replace("²", "2").replace("/", "_")
                .replace("(", "").replace(")", "").replace("%", "pct")
                .replace("-", "_"))
        out = os.path.join(output_dir, f"{cut_code}_{safe}.png")
        make_individual_chart(df, feat, short_names[feat], features[feat],
                              target, global_mean, out)

    # 2. Combined chart
    print(f"\n  Building combined continuous chart...")
    combined_out = os.path.join(output_dir, f"{cut_code}_continuous_features.png")
    make_combined_chart(df, feat_list, short_names, features, target,
                        global_mean, cut_label, n_label, combined_out)

    # 3. Excel
    print(f"\n  Building Excel workbook...")
    excel_out = os.path.join(output_dir, f"{cut_code}_Feature_Analysis_Tables.xlsx")
    make_excel(df, feat_list, short_names, features, insights,
               target, global_mean, cut_label, excel_out)

    print(f"\n  ✓ All outputs saved to {output_dir}")


# ═══════════════════════════════════════════════════════════════════════════
# ══  CONFIG — edit this section to run for a new cut  ═════════════════════
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Example: Fillet (FT) ─────────────────────────────────────────────
    INPUT_PATH  = "/mnt/user-data/uploads/ALL_FT_meat_8_1.csv"
    OUTPUT_DIR  = "/mnt/user-data/outputs"
    CUT_CODE    = "FT"
    CUT_LABEL   = "Fillet (FT)"
    TARGET      = "EQ"

    # Features: {column_name: importance_pct}
    FEATURES = {
        "kill_month":                                   10.14,
        "Subjective - MSA Eye Muscle Area (cm2)":        6.85,
        "Objective - QFOM -  Meat Colour":               3.62,
        "Subjective - MSA Hump Height (mm)":             3.46,
        "48hr - Fillet -   temp":                        3.08,
        "Conductivity - 21days -  Fillet -  Cut face":   2.87,
        "Subjective - USDA Eye Muscle Area (in2)":       2.71,
        "Cold Weight (kg)":                              2.51,
        "Subjective - MSA Marbling Score":               2.20,
    }

    # Short display names: {column_name: label_for_charts}
    SHORT_NAMES = {
        "kill_month":                                  "Kill Month",
        "Subjective - MSA Eye Muscle Area (cm2)":      "MSA Eye Muscle Area (cm²)",
        "Objective - QFOM -  Meat Colour":             "QFOM Meat Colour",
        "Subjective - MSA Hump Height (mm)":           "MSA Hump Height (mm)",
        "48hr - Fillet -   temp":                      "48hr Fillet Temp",
        "Conductivity - 21days -  Fillet -  Cut face": "Conductivity 21d Cut Face",
        "Subjective - USDA Eye Muscle Area (in2)":     "USDA Eye Muscle Area (in²)",
        "Cold Weight (kg)":                            "Cold Weight (kg)",
        "Subjective - MSA Marbling Score":             "MSA Marbling Score",
    }

    # Insights: {column_name: [observation_1, observation_2]}
    # Leave as empty list [] if you have no observations yet for a feature.
    INSIGHTS = {
        "kill_month": [
            "Clear seasonal pattern — animals killed May–December score consistently "
            "higher than those killed January–April, with December the strongest month "
            "(mean EQ 7.20) and April the weakest (6.42). A spread of ~0.77 points "
            "across the year.",
            "Kill month is the highest importance feature for Fillet (10.14%). The "
            "seasonality likely reflects pasture quality cycles, with spring-killed "
            "animals at the end of a tougher winter finishing period performing worst.",
        ],
        "Subjective - MSA Eye Muscle Area (cm2)": [
            "Negative relationship (r = −0.192) — larger eye muscle area is associated "
            "with lower EQ. The Q1 (low EQ) group has a notably higher mean eye muscle "
            "area than the Q4 (high EQ) group.",
            "This reflects the known muscle hypertrophy effect: animals with very large "
            "eye muscles tend to have coarser muscle fibres and less intramuscular fat, "
            "both of which reduce eating quality.",
        ],
        "Objective - QFOM -  Meat Colour": [
            "Weak positive relationship (r = +0.069). Higher QFOM colour scores "
            "(brighter, redder meat) are associated with marginally better EQ across "
            "all four quartiles.",
            "The effect is small and the spread within each quartile is wide, suggesting "
            "meat colour alone is not a reliable EQ predictor for Fillet.",
        ],
        "Subjective - MSA Hump Height (mm)": [
            "Negative relationship (r = −0.166) — taller hump height is associated with "
            "lower EQ. Animals with smaller humps (more British/European type) score "
            "better on average.",
            "Hump height is a proxy for Bos indicus influence. Higher hump heights "
            "indicate more Zebu genetics, which is associated with elevated calpastatin "
            "activity and reduced proteolytic tenderisation during ageing.",
        ],
        "48hr - Fillet -   temp": [
            "Very weak negative relationship (r = −0.040, p = 0.045) — marginally "
            "significant but of negligible practical size.",
            "Despite its CatBoost importance (3.08%), the linear signal is minimal. "
            "CatBoost is likely capturing a threshold or interaction effect rather "
            "than a simple linear trend.",
        ],
        "Conductivity - 21days -  Fillet -  Cut face": [
            "No meaningful relationship (r = −0.005, p = 0.889). The feature is "
            "essentially uncorrelated with EQ in the Fillet dataset.",
            "This feature also has the highest missingness rate — only 672 of 2,700 "
            "rows are valid (75% missing). Both the weak signal and poor coverage "
            "suggest limited value for Fillet modelling.",
        ],
        "Subjective - USDA Eye Muscle Area (in2)": [
            "Negative relationship (r = −0.223) — corrected from r = −0.100 after a "
            "data entry error (96 instead of 9.6 for animal 12143) was fixed. Clean "
            "range is now 7.10–19.40 in².",
            "The two eye muscle area features (MSA cm² and USDA in²) carry overlapping "
            "information. The corrected USDA signal is now the stronger of the two "
            "for this cut.",
        ],
        "Cold Weight (kg)": [
            "The strongest linear predictor among continuous features for Fillet "
            "(r = −0.200). Heavier carcasses score lower on EQ — Q1 mean cold weight "
            "is 340 kg versus 307 kg for Q4, a 33 kg difference.",
            "Cold weight is partly a proxy for breed, age, and production system. "
            "Heavier Continental-cross animals typically have larger muscles and lower "
            "intramuscular fat, both of which reduce eating quality.",
        ],
        "Subjective - MSA Marbling Score": [
            "Positive relationship (r = +0.121) — the only continuous feature with a "
            "meaningful positive direction. Higher marbling is consistently associated "
            "with better EQ.",
            "For Fillet specifically, where intramuscular fat is naturally lower than "
            "in Loin, even modest marbling differences appear to matter.",
        ],
    }

    run_analysis(
        input_path  = INPUT_PATH,
        output_dir  = OUTPUT_DIR,
        cut_code    = CUT_CODE,
        cut_label   = CUT_LABEL,
        features    = FEATURES,
        short_names = SHORT_NAMES,
        insights    = INSIGHTS,
        target      = TARGET,
    )

In [ ]:
INPUT_PATH  = "/path/to/ALL_SM_meat_8_1.csv"
OUTPUT_DIR  = "/path/to/outputs"
CUT_CODE    = "SM"                  # used in filenames
CUT_LABEL   = "Topside (SM)"        # used in chart titles
TARGET      = "EQ"

FEATURES = {
    "column_name": importance_pct,  # one entry per feature
    ...
}

SHORT_NAMES = {
    "column_name": "Display Label",  # short label for axes/titles
    ...
}

INSIGHTS = {
    "column_name": [
        "First observation.",
        "Second observation.",
    ],
    ...
}